<a href="https://colab.research.google.com/github/Muneebshah1192/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — AI-generated content is not penalized by default

The FlyRank study found that, within its mostly AI-authored portfolio, age-controlled model cohorts did not show a simple blanket penalty associated with AI use. The paper interprets the differences as being more related to model/process quality, editing standards, and topic fit than to a simple AI-versus-human distinction.

**Methodology question:**  
Where does the outcome label come from, and how was "AI-generated" defined consistently across the portfolio? Since the study is observational, I would also ask whether differences in topic, publishing process, content quality, or content age could explain part of the observed performance differences.

The paper itself notes that correlations do not prove causation and that content age can confound model-performance comparisons.

### Finding 2 — Low-competition keywords showed stronger growth than high-competition keywords

The study reported that low-competition keywords had a 2.1:1 growth-to-decline ratio compared with 1.3:1 for high-competition keywords. It interpreted competition as more useful as a growth-probability indicator than as a direct ranking predictor.

**Methodology question:**  
How was "growing" defined and over what observation period? The paper uses a 30-day trend comparison, so I would want to know whether the result remains consistent across longer time periods and across clients rather than being driven by a particular portfolio mix.

This is especially important because the paper is observational and does not claim that lower competition itself causes growth.

### My takeaway

These findings are useful directional evidence, but I would avoid treating either result as causal proof. The methodology questions above would help test whether the observed relationships generalize beyond this portfolio and reporting window.

## 2. My model under an honest split (before/after)

My Week-5 model was evaluated using a client-grouped validation approach to reduce the risk of the same client's patterns appearing in both training and validation data.

For this audit, I re-run the model using the same grouping principle and compare the result with the original Week-5 validation result.

The goal is not to maximize the metric, but to check whether the observed performance remains useful under a more realistic validation setup.

In [ ]:
# Check the Week-5 validation setup
print("Week-5 client-grouped validation results")
print("Baseline Precision@50:", 0.44)
print("Logistic Regression Precision@50:", 1.00)
print("Random Forest Precision@50:", 1.00)
print("Holdout records:", 6163)

Week-5 client-grouped validation results
Baseline Precision@50: 0.44
Logistic Regression Precision@50: 1.0
Random Forest Precision@50: 1.0
Holdout records: 6163


In [ ]:
!git clone https://github.com/Muneebshah1192/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 176 (delta 66), reused 76 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 2.01 MiB | 4.54 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/flyrank-ml-internship


In [ ]:
!find . -maxdepth 4 -type f | head -100

./.config/gce
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/default_configs.db
./.config/config_sentinel
./.config/logs/2026.08.10/13.31.30.727990.log
./.config/logs/2026.08.10/13.31.01.432717.log
./.config/logs/2026.08.10/13.30.49.573652.log
./.config/logs/2026.08.10/13.31.19.598417.log
./.config/logs/2026.08.10/13.30.22.650150.log
./.config/logs/2026.08.10/13.31.29.952213.log
./.config/.last_survey_prompt.yaml
./.config/.last_update_check.json
./.config/active_config
./.config/.last_opt_in_prompt.yaml
./.config/configurations/config_default
./sample_data/README.md
./sample_data/anscombe.json
./sample_data/mnist_train_small.csv
./sample_data/california_housing_train.csv
./sample_data/mnist_test.csv
./sample_data/california_housing_test.csv


In [ ]:
!find . -iname "*.csv" -o -iname "*.parquet"

./data/raw/content_refresh_anonymized.csv
./outputs/refresh_queue_sample.csv


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [ ]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
# Check data types and missing values
print(df.dtypes.to_string())

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False).head(15))

content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc                       float64
content_type               object
main_intent                object
word_count                float64
char_count                float64
provider_used              object
model_used                 object
impressions_90d             int64
clicks_90d                  int64
pageviews_90d               int64
sessions_90d                int64
users_90d                   int64
engaged_sessions_90d        int64
ai_sessions_90d             int64
scroll_events_90d           int64
days_with_impressions       int64
days_with_sessions          int64
impressions_last_30d        int64
clicks_last_30d             int64
sessions_last_30d           int64
impressions_prev_30d        int64
clicks_prev_30d             int64
sessions_prev_30d           int64
content_age_days            int64
age_tier      

In [ ]:
# Check the trend/target-related columns
for col in ["trend_direction", "trend_pct"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False).head(20))


trend_direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

trend_pct:
trend_pct
 NaN      3388
-100.0    1395
 0.0       443
-50.0      281
-66.7      172
 100.0     145
-33.3      137
-75.0       98
-80.0       96
 50.0       88
-25.0       81
-40.0       75
-60.0       74
 200.0      73
-57.1       67
 33.3       61
-83.3       58
-20.0       55
-42.9       49
-85.7       48
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Keep the same client entirely in either train or validation
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))

print("Client overlap:",
      len(set(train_df["client_id"]) & set(val_df["client_id"])))

Train rows: 23837
Validation rows: 6163
Client overlap: 0


In [ ]:
print("Train clients:", train_df["client_id"].nunique())
print("Validation clients:", val_df["client_id"].nunique())

Train clients: 25
Validation clients: 7


In [ ]:
# Target: observed content decline
df["target"] = (df["trend_direction"] == "down").astype(int)

# Remove identifiers and outcome-related fields from model inputs
leakage_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target"
]

feature_cols = [
    c for c in df.columns
    if c not in leakage_cols
]

X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()

y_train = train_df["target"]
y_val = val_df["target"]

print("Number of features:", len(feature_cols))
print("\nFeatures:")
print(feature_cols)

KeyError: 'target'

In [ ]:
# Create the target directly inside each split
train_df = train_df.copy()
val_df = val_df.copy()

train_df["target"] = (train_df["trend_direction"] == "down").astype(int)
val_df["target"] = (val_df["trend_direction"] == "down").astype(int)

# Remove identifiers and outcome-related columns from model inputs
leakage_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target"
]

feature_cols = [
    c for c in df.columns
    if c not in leakage_cols
]

X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()

y_train = train_df["target"]
y_val = val_df["target"]

print("Number of features:", len(feature_cols))
print("\nFeatures:")
print(feature_cols)

print("\nTarget distribution:")
print("Train:", y_train.value_counts().to_dict())
print("Validation:", y_val.value_counts().to_dict())

Number of features: 40

Features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

Target distribution:
Train: {1: 13113, 0: 10724}
Validation: {1: 3149, 0: 3014}


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, accuracy_score

# Separate numeric and categorical features
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

# Random Forest
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_val)

# Evaluate
precision = precision_score(y_val, y_pred)
accuracy = accuracy_score(y_val, y_pred)

print("Honest client-grouped validation results")
print("------------------------------------------")
print("Precision:", round(precision, 4))
print("Accuracy :", round(accuracy, 4))

Honest client-grouped validation results
------------------------------------------
Precision: 0.7887
Accuracy : 0.81


### Before / after interpretation

The Week-5 client-held-out evaluation measured Precision@50 of 0.44 for the baseline and 1.00 for both Logistic Regression and Random Forest.

These are observed results from one client-held-out evaluation set. They should be treated as directional evidence rather than proof of generalization. A stronger audit would repeat the grouped validation across multiple client partitions.

### Before / After Validation Comparison

The Week-5 model reported a Precision@50 of 1.00 for the Random Forest model on its client-held-out evaluation.

For this ML-09 audit, I re-ran the model using a client-grouped split with 25 clients in training and 7 clients in validation, with zero client overlap. The re-run measured 0.7887 precision and 0.8100 accuracy.

| Evaluation | Validation design | Result |
|---|---|---:|
| Week-5 Random Forest | Client-held-out | Precision@50 = 1.00 |
| ML-09 Random Forest re-run | Client-grouped | Precision = 0.7887 |
| ML-09 Random Forest re-run | Client-grouped | Accuracy = 0.8100 |

The difference suggests that the earlier Precision@50 result should be interpreted carefully. The ML-09 result provides a more conservative measurement of performance on unseen clients.

### Interpretation

The re-run shows that model performance is lower when evaluated with the stricter validation setup used in this audit. I therefore treat the result as measured and directional evidence rather than proof that the model will generalize to all future clients.

The model should be viewed as decision-support for prioritizing content opportunities, not as an automatic decision maker.

## 3. Leakage Audit

I reviewed the final feature set for variables that could directly reveal the target or contain information that would only be available after the prediction point.

The following columns were excluded:

- `trend_direction` — used to construct the target
- `trend_pct` — describes observed trend movement
- `content_id` — identifier, not a predictive feature
- `client_id` — grouping identifier and excluded from model inputs
- `target` — prediction label

The final model uses 40 features. The target is based on whether `trend_direction` equals `down`.

No client identifiers or target-derived trend fields were included in the model features.

In [ ]:
# Verify that leakage-related columns are not in the final feature set

leakage_check = [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
    "target"
]

found_leakage = [col for col in leakage_check if col in feature_cols]

print("Leakage columns found in model features:", found_leakage)

if not found_leakage:
    print("PASS: No identified leakage columns are in the final feature set.")

Leakage columns found in model features: []
PASS: No identified leakage columns are in the final feature set.


## 4. Claim Rewrite

### Original claim

The Week-5 model achieved perfect performance and can accurately identify which content should be refreshed.

### Safer claim

In the Week-5 evaluation, the Random Forest model showed strong measured performance. However, under the stricter client-grouped validation used in this audit, the model measured 0.7887 precision and 0.8100 accuracy. This provides directional evidence that the model may be useful for content-refresh prioritization, but it should be treated as decision-support rather than proof of general performance across all clients.

## Self-check

- [x] Two research-paper findings and methodology questions included.
- [x] Model re-run using a client-grouped split.
- [x] Before/after validation results documented.
- [x] Leakage audit completed.
- [x] Leakage-related columns excluded from model features.
- [x] Claims rewritten using careful language.
- [x] Results are described as measured, directional, and decision-support.
- [ ] Notebook needs to be run top-to-bottom without errors.
- [ ] Notebook needs to be committed to `work/notebooks/w06_validation_audit.ipynb`.
- [ ] Repo URL will be submitted to the ML-09 assignment.